# Notebook 05 — Training
## Adaptive Reliability-Aware Fusion (ARAF) Project

**Goal of this notebook:**
Train all four models (UnimodalImage, UnimodalText, NaiveFusion, ARAF)
using corruption-aware training, then save checkpoints for evaluation.

**What corruption-aware training means:**
Every batch goes through the CorruptionModule before hitting the model.
The model never sees only clean data during training — it always sees
a mix of clean, corrupted, and missing-modality samples. This is what
makes ARAF robust at test time.

**What you will learn:**
- How to write a clean PyTorch training loop
- What corruption-aware training looks like in practice
- How to track and plot training curves
- How learning rate scheduling works
- How to save and load model checkpoints
- What healthy vs unhealthy training looks like

**Important note on training time:**
With frozen encoders and 2000 samples, one epoch takes roughly:
- CPU: 5-10 minutes
- GPU: 30-60 seconds

We train for 5 epochs by default. Adjust `NUM_EPOCHS` based on your hardware.
For a paper you would train for 20-30 epochs on the full dataset.

---


## 1. Imports and setup


In [ ]:
import os, sys, json, copy, random, time
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import BertTokenizer
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Constants ─────────────────────────────────────────────────────────────────
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
IMAGE_SIZE     = 224
MAX_TEXT_LEN   = 32

clean_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Imports successful.")


## 2. Reload all components

From this notebook onward, all logic lives in `.py` files.
The notebook just imports and orchestrates.


In [ ]:
# ── MultimodalSample ──────────────────────────────────────────────────────────
@dataclass
class MultimodalSample:
    image: torch.Tensor
    text_ids: torch.Tensor
    attention_mask: torch.Tensor
    label: torch.Tensor
    raw_image: Optional[object] = None
    raw_text: str = ""
    dataset_name: str = "vqa_v2"
    sample_id: str = ""
    image_corrupted: bool = False
    text_corrupted: bool = False
    image_missing: bool = False
    text_missing: bool = False
    corruption_severity: float = 0.0

import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

baselines   = load_module("baselines",   "models/baselines.py")
araf_module = load_module("araf",        "models/araf.py")
corr_module = load_module("corruption",  "corruption/corruption_module.py")

ImageEncoder       = baselines.ImageEncoder
TextEncoder        = baselines.TextEncoder
ClassificationHead = baselines.ClassificationHead
UnimodalImageModel = baselines.UnimodalImageModel
UnimodalTextModel  = baselines.UnimodalTextModel
NaiveFusionModel   = baselines.NaiveFusionModel
vqa_loss           = baselines.vqa_loss
vqa_accuracy       = baselines.vqa_accuracy

ARAFModel          = araf_module.ARAFModel
CorruptionModule   = corr_module.CorruptionModule

print("All modules loaded.")


## 3. Dataset and DataLoader

We build a proper PyTorch Dataset that applies corruption on-the-fly
inside `__getitem__`. This means every time a sample is loaded,
it gets a freshly randomized corruption — so the model sees
different corruptions of the same image across different epochs.

### Why apply corruption inside the Dataset?

If we applied corruption once before training, the model would
memorize the specific corrupted versions rather than learning
to handle corruption in general. On-the-fly corruption acts as
data augmentation — each epoch the model sees slightly different
degradations, which forces generalization.


In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
with open("answer_vocab.json") as f:
    answer2idx = json.load(f)
idx2answer  = {v: k for k, v in answer2idx.items()}
NUM_CLASSES = len(answer2idx)

print("Loading VQA v2...")
hf_val_full = load_dataset("lmms-lab/VQAv2", split="validation",
                           trust_remote_code=True)
split    = hf_val_full.train_test_split(test_size=0.2, seed=42)
hf_train = split["train"]
hf_val   = split["test"]
print(f"Train: {len(hf_train):,}  Val: {len(hf_val):,}")


class VQACorruptionDataset(Dataset):
    """
    PyTorch Dataset that applies corruption on-the-fly.

    Each __getitem__ call:
    1. Loads the raw image and question
    2. Applies a random corruption (or none, with probability 1-p_corrupt)
    3. Returns a batch-ready dict

    This means every epoch the model sees different corruptions
    of the same samples — corruption as data augmentation.

    Args:
        hf_split         : HuggingFace dataset split
        tokenizer        : BertTokenizer
        answer2idx       : answer vocabulary
        corruption_module: CorruptionModule instance (or None for clean)
        max_samples      : limit dataset size for development
        num_classes      : vocabulary size
    """
    def __init__(self, hf_split, tokenizer, answer2idx,
                 corruption_module=None, max_samples=None, num_classes=3129):
        self.data     = hf_split
        self.tok      = tokenizer
        self.a2i      = answer2idx
        self.corr     = corruption_module
        self.n_cls    = num_classes

        if max_samples:
            self.data = self.data.select(range(min(max_samples, len(self.data))))
        print(f"VQACorruptionDataset: {len(self.data):,} samples, "
              f"corruption={'ON' if corruption_module else 'OFF'}")

    def __len__(self): return len(self.data)

    def _make_label(self, answers):
        label = torch.zeros(self.n_cls)
        cnt = Counter(a["answer"].lower().strip() for a in answers)
        for ans, c in cnt.items():
            if ans in self.a2i:
                label[self.a2i[ans]] = min(c / 3.0, 1.0)
        return label

    def __getitem__(self, idx):
        row = self.data[idx]
        pil = row["image"].convert("RGB")
        img = clean_transform(pil)
        enc = self.tok(row["question"], padding="max_length",
                       max_length=MAX_TEXT_LEN, truncation=True,
                       return_tensors="pt")
        sample = MultimodalSample(
            image=img,
            text_ids=enc["input_ids"].squeeze(0),
            attention_mask=enc["attention_mask"].squeeze(0),
            label=self._make_label(row["answers"]),
            raw_text=row["question"],
            sample_id=str(row.get("question_id", idx)),
        )
        if self.corr:
            sample = self.corr(sample)
        return {
            "image"             : sample.image,
            "text_ids"          : sample.text_ids,
            "attention_mask"    : sample.attention_mask,
            "label"             : sample.label,
            "image_corrupted"   : sample.image_corrupted,
            "text_corrupted"    : sample.text_corrupted,
            "image_missing"     : sample.image_missing,
            "text_missing"      : sample.text_missing,
            "corruption_severity": sample.corruption_severity,
            "raw_text"          : sample.raw_text,
            "sample_id"         : sample.sample_id,
        }


In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# Adjust these based on your hardware and time budget.
# For a first run: keep these small to verify everything works end-to-end.
# For paper results: increase MAX_TRAIN_SAMPLES and NUM_EPOCHS significantly.

MAX_TRAIN_SAMPLES = 2000   # set to None for full ~200k dataset
MAX_VAL_SAMPLES   = 500
BATCH_SIZE        = 16     # reduce to 8 if you get OOM errors
NUM_EPOCHS        = 5      # increase to 20-30 for paper results
LEARNING_RATE     = 3e-4   # Adam default for classification heads
LAMBDA_REG        = 0.1    # reliability regularization weight

print(f"Training config:")
print(f"  MAX_TRAIN_SAMPLES : {MAX_TRAIN_SAMPLES}")
print(f"  MAX_VAL_SAMPLES   : {MAX_VAL_SAMPLES}")
print(f"  BATCH_SIZE        : {BATCH_SIZE}")
print(f"  NUM_EPOCHS        : {NUM_EPOCHS}")
print(f"  LEARNING_RATE     : {LEARNING_RATE}")
print(f"  LAMBDA_REG        : {LAMBDA_REG}")
print(f"  DEVICE            : {DEVICE}")

# ── Corruption module for training ────────────────────────────────────────────
# These probabilities produce a balanced mix of:
#   ~25% clean, ~25% image-only corrupted,
#   ~25% text-only corrupted, ~25% both corrupted
# plus ~10% missing modality cases
train_corruption = CorruptionModule(
    p_corrupt_image = 0.5,
    p_corrupt_text  = 0.5,
    p_missing_image = 0.1,
    p_missing_text  = 0.1,
    severity        = None,  # random severity 1-5 each sample
)

# Validation uses NO corruption (we evaluate clean performance here)
# We evaluate corrupted performance separately in Notebook 06
val_corruption = None

# ── Build datasets ────────────────────────────────────────────────────────────
train_dataset = VQACorruptionDataset(
    hf_train, tokenizer, answer2idx,
    corruption_module=train_corruption,
    max_samples=MAX_TRAIN_SAMPLES,
    num_classes=NUM_CLASSES,
)
val_dataset = VQACorruptionDataset(
    hf_val, tokenizer, answer2idx,
    corruption_module=val_corruption,
    max_samples=MAX_VAL_SAMPLES,
    num_classes=NUM_CLASSES,
)

# ── Build DataLoaders ─────────────────────────────────────────────────────────
def collate_fn(batch):
    """Stack tensors, keep metadata as lists."""
    return {
        "image"             : torch.stack([b["image"] for b in batch]),
        "text_ids"          : torch.stack([b["text_ids"] for b in batch]),
        "attention_mask"    : torch.stack([b["attention_mask"] for b in batch]),
        "label"             : torch.stack([b["label"] for b in batch]),
        "image_corrupted"   : [b["image_corrupted"] for b in batch],
        "text_corrupted"    : [b["text_corrupted"] for b in batch],
        "image_missing"     : [b["image_missing"] for b in batch],
        "text_missing"      : [b["text_missing"] for b in batch],
        "corruption_severity": [b["corruption_severity"] for b in batch],
        "raw_text"          : [b["raw_text"] for b in batch],
        "sample_id"         : [b["sample_id"] for b in batch],
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, collate_fn=collate_fn)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


## 4. Build all four models

We train all models with the same setup so comparison is fair:
- Same encoder (frozen ResNet-50 + frozen BERT)
- Same optimizer (Adam)
- Same learning rate
- Same training data and corruption

The only difference is the fusion strategy.


In [ ]:
def build_models():
    """Build all four models fresh with random classification head weights."""
    img_model    = UnimodalImageModel(num_classes=NUM_CLASSES,
                                      frozen_encoder=True).to(DEVICE)
    txt_model    = UnimodalTextModel(num_classes=NUM_CLASSES,
                                     frozen_encoder=True).to(DEVICE)
    fusion_model = NaiveFusionModel(num_classes=NUM_CLASSES,
                                    frozen_encoders=True).to(DEVICE)
    araf_model   = ARAFModel(num_classes=NUM_CLASSES, fusion_dim=512,
                             frozen_encoders=True, lambda_reg=LAMBDA_REG).to(DEVICE)
    return {
        "UnimodalImage" : img_model,
        "UnimodalText"  : txt_model,
        "NaiveFusion"   : fusion_model,
        "ARAF"          : araf_model,
    }

models = build_models()

print("Models built:")
for name, model in models.items():
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {name:<20} trainable params: {trainable:>10,}")


## 5. Optimizer and learning rate scheduler

### Why Adam?

Adam (Adaptive Moment Estimation) is the standard optimizer for
fine-tuning pretrained models. It adapts the learning rate per parameter,
which works well when different parts of the model (frozen encoder vs
trainable head) have very different gradient magnitudes.

### Why a scheduler?

A constant learning rate often overshoots the minimum in later epochs.
We use `CosineAnnealingLR` which starts at `LEARNING_RATE` and smoothly
decays to near-zero over training. This is the most common scheduler
in multimodal research papers.

```
LR
 |  start high
 |   \
 |    \
 |     \___
 |          \_____ end near zero
 +──────────────── epochs
```

### One optimizer per model

Each model trains independently with its own optimizer and scheduler.
This ensures fair comparison — no model benefits from another's gradients.


In [ ]:
def build_optimizers(models):
    """Build one Adam optimizer + cosine scheduler per model."""
    optimizers = {}
    schedulers = {}
    for name, model in models.items():
        # Only optimize trainable parameters (skip frozen encoder weights)
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.Adam(trainable_params, lr=LEARNING_RATE,
                               weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=NUM_EPOCHS, eta_min=1e-6)
        optimizers[name] = opt
        schedulers[name] = sch
    return optimizers, schedulers

optimizers, schedulers = build_optimizers(models)
print("Optimizers and schedulers built.")
for name, opt in optimizers.items():
    print(f"  {name}: {len(opt.param_groups[0]['params'])} param groups, "
          f"lr={opt.param_groups[0]['lr']}")


## 6. Single training step

The training step is the innermost loop. It runs once per batch:

```
1. Move batch to device
2. Forward pass through model
3. Compute loss
4. Zero gradients
5. Backward pass (compute gradients)
6. Clip gradients (prevent exploding gradients)
7. Optimizer step (update weights)
8. Return loss and accuracy for logging
```

### Why gradient clipping?

With large models (BERT has 110M params), gradients can occasionally
become very large and cause weight updates that destroy what the model
has learned (exploding gradient problem). Clipping ensures no gradient
vector has a norm larger than `max_norm=1.0`. This is standard practice.


In [ ]:
def train_step(model, batch, optimizer, model_name: str) -> Dict:
    """
    Single training step for one batch.

    Args:
        model      : one of our four models
        batch      : dict from DataLoader
        optimizer  : Adam optimizer for this model
        model_name : used to call the right forward/loss interface

    Returns:
        dict with loss values and accuracy
    """
    model.train()

    # ── Forward pass ──────────────────────────────────────────────────────────
    if model_name == "ARAF":
        output = model(batch)
        losses = model.compute_loss(batch, output)
        loss   = losses["total_loss"]
        task_l = losses["task_loss"].item()
        reg_l  = losses["reg_loss"].item()
    else:
        # Baselines only have task loss
        output = model(batch)
        labels = batch["label"].to(DEVICE)
        loss   = vqa_loss(output["logits"], labels)
        task_l = loss.item()
        reg_l  = 0.0

    # ── Backward pass ─────────────────────────────────────────────────────────
    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping — prevents exploding gradients
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad],
        max_norm=1.0
    )

    optimizer.step()

    # ── Accuracy ──────────────────────────────────────────────────────────────
    with torch.no_grad():
        labels = batch["label"].to(DEVICE)
        acc = vqa_accuracy(output["logits"].detach(), labels)

    return {
        "total_loss": loss.item(),
        "task_loss" : task_l,
        "reg_loss"  : reg_l,
        "accuracy"  : acc,
    }


def val_step(model, batch, model_name: str) -> Dict:
    """
    Single validation step. No gradient computation.
    """
    model.eval()
    with torch.no_grad():
        if model_name == "ARAF":
            output = model(batch)
            losses = model.compute_loss(batch, output)
            loss   = losses["total_loss"].item()
        else:
            output = model(batch)
            labels = batch["label"].to(DEVICE)
            loss   = vqa_loss(output["logits"], labels).item()

        labels = batch["label"].to(DEVICE)
        acc    = vqa_accuracy(output["logits"], labels)

    return {"loss": loss, "accuracy": acc}


print("Training and validation step functions defined.")


## 7. Full training loop

The training loop iterates over epochs. Each epoch:
1. Trains on all batches in `train_loader` (with corruption)
2. Evaluates on all batches in `val_loader` (clean)
3. Steps the learning rate scheduler
4. Saves the best checkpoint (based on val accuracy)
5. Prints a progress summary

We train all four models in parallel (one after the other per batch)
so they see exactly the same corrupted data for fair comparison.

### What to watch during training

- **Loss should decrease** over epochs. If it goes up, something is wrong.
- **Accuracy should increase**. With 5 epochs on 2000 samples expect ~0.20-0.40.
- **ARAF reg_loss** should also decrease — this means the reliability
  estimator is learning to identify corrupted inputs.
- **ARAF should outperform NaiveFusion** especially in later epochs.


In [ ]:
# ── History tracking ──────────────────────────────────────────────────────────
history = {name: {
    "train_loss": [], "train_acc": [],
    "val_loss":   [], "val_acc":  [],
    "reg_loss":   [],   # ARAF only
} for name in models}

best_val_acc  = {name: 0.0 for name in models}
best_ckpt_path= {name: f"checkpoints/{name}_best.pt" for name in models}
os.makedirs("checkpoints", exist_ok=True)

print("Starting training...")
print("=" * 70)

total_start = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    # ── Per-model epoch accumulators ──────────────────────────────────────────
    train_stats = {name: defaultdict(list) for name in models}
    val_stats   = {name: defaultdict(list) for name in models}

    # ── Training phase ────────────────────────────────────────────────────────
    for batch_idx, batch in enumerate(train_loader):
        for name, model in models.items():
            step_out = train_step(model, batch, optimizers[name], name)
            for k, v in step_out.items():
                train_stats[name][k].append(v)

        # Progress print every 20 batches
        if (batch_idx + 1) % 20 == 0:
            araf_loss = np.mean(train_stats["ARAF"]["total_loss"])
            araf_acc  = np.mean(train_stats["ARAF"]["accuracy"])
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS} "
                  f"Batch {batch_idx+1}/{len(train_loader)} | "
                  f"ARAF loss={araf_loss:.4f} acc={araf_acc:.4f}")

    # ── Validation phase ──────────────────────────────────────────────────────
    for batch in val_loader:
        for name, model in models.items():
            step_out = val_step(model, batch, name)
            for k, v in step_out.items():
                val_stats[name][k].append(v)

    # ── Step schedulers ───────────────────────────────────────────────────────
    for name in models:
        schedulers[name].step()

    # ── Log epoch stats ───────────────────────────────────────────────────────
    epoch_time = time.time() - epoch_start
    print(f"
Epoch {epoch+1}/{NUM_EPOCHS} completed in {epoch_time:.1f}s")
    print(f"{'Model':<20} {'Train Loss':>10} {'Train Acc':>10} "
          f"{'Val Loss':>10} {'Val Acc':>10}")
    print("-" * 60)

    for name in models:
        tr_loss = np.mean(train_stats[name]["total_loss"])
        tr_acc  = np.mean(train_stats[name]["accuracy"])
        vl_loss = np.mean(val_stats[name]["loss"])
        vl_acc  = np.mean(val_stats[name]["accuracy"])
        reg_l   = np.mean(train_stats[name]["reg_loss"]) if name == "ARAF" else 0.0

        history[name]["train_loss"].append(tr_loss)
        history[name]["train_acc"].append(tr_acc)
        history[name]["val_loss"].append(vl_loss)
        history[name]["val_acc"].append(vl_acc)
        history[name]["reg_loss"].append(reg_l)

        print(f"{name:<20} {tr_loss:>10.4f} {tr_acc:>10.4f} "
              f"{vl_loss:>10.4f} {vl_acc:>10.4f}")

        # Save best checkpoint
        if vl_acc > best_val_acc[name]:
            best_val_acc[name] = vl_acc
            torch.save({
                "epoch"     : epoch + 1,
                "model_name": name,
                "state_dict": model.state_dict(),
                "val_acc"   : vl_acc,
                "val_loss"  : vl_loss,
            }, best_ckpt_path[name])
            print(f"  -> New best for {name}: val_acc={vl_acc:.4f} (saved)")

    print()

total_time = time.time() - total_start
print(f"Training complete in {total_time/60:.1f} minutes.")
print("Best validation accuracies:")
for name, acc in best_val_acc.items():
    print(f"  {name:<20}: {acc:.4f}")


## 8. Training curves visualization

Training curves are the first thing your advisor and reviewers will
look at. They show whether training was stable, whether models converged,
and whether there is overfitting (val loss going up while train loss goes down).

### What healthy training curves look like

- Train loss: smoothly decreasing
- Val loss: decreasing, staying close to train loss
- Train acc: smoothly increasing
- Val acc: increasing, close to train acc

### Signs of problems

- Loss increases: learning rate too high, or a bug
- Val loss diverges from train loss: overfitting (need more data or regularization)
- Flat curves: learning rate too low, or gradients not flowing


In [ ]:
def plot_training_curves(history, models_to_plot=None):
    """Plot loss and accuracy curves for all models."""
    if models_to_plot is None:
        models_to_plot = list(history.keys())

    colors = {
        "UnimodalImage" : "#7F77DD",
        "UnimodalText"  : "#1D9E75",
        "NaiveFusion"   : "#D85A30",
        "ARAF"          : "#185FA5",
    }
    epochs = list(range(1, NUM_EPOCHS + 1))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── Plot 1: Training loss ─────────────────────────────────────────────────
    for name in models_to_plot:
        axes[0].plot(epochs, history[name]["train_loss"],
                     color=colors[name], label=name, linewidth=2)
    axes[0].set_title("Training loss", fontsize=12)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)

    # ── Plot 2: Validation accuracy ───────────────────────────────────────────
    for name in models_to_plot:
        axes[1].plot(epochs, history[name]["val_acc"],
                     color=colors[name], label=name, linewidth=2, marker="o")
    axes[1].set_title("Validation accuracy (clean)", fontsize=12)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("VQA accuracy")
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    # ── Plot 3: ARAF loss decomposition ──────────────────────────────────────
    axes[2].plot(epochs, history["ARAF"]["train_loss"],
                 color="#185FA5", label="Total loss", linewidth=2)
    # Reconstruct task loss from total - reg
    task_losses = [t - r for t, r in zip(history["ARAF"]["train_loss"],
                                          history["ARAF"]["reg_loss"])]
    axes[2].plot(epochs, task_losses,
                 color="#185FA5", label="Task loss", linewidth=2, linestyle="--")
    axes[2].plot(epochs, history["ARAF"]["reg_loss"],
                 color="#FAC775", label="Reg loss", linewidth=2, linestyle=":")
    axes[2].set_title("ARAF loss decomposition", fontsize=12)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Loss")
    axes[2].legend(fontsize=9)
    axes[2].grid(alpha=0.3)

    plt.suptitle("Training curves — all models", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: training_curves.png")

plot_training_curves(history)


## 9. Final validation summary

A clean table of results at the end of training.
This is the starting point for your results section.


In [ ]:
print("=" * 60)
print("FINAL TRAINING SUMMARY")
print("=" * 60)
print(f"{'Model':<20} {'Best Val Acc':>12} {'Final Val Acc':>14}")
print("-" * 60)
for name in models:
    best = best_val_acc[name]
    final = history[name]["val_acc"][-1]
    marker = " <-- best" if best == max(best_val_acc.values()) else ""
    print(f"{name:<20} {best:>12.4f} {final:>14.4f}{marker}")
print("=" * 60)
print()
print("Checkpoints saved:")
for name, path in best_ckpt_path.items():
    if os.path.exists(path):
        ckpt = torch.load(path, map_location="cpu")
        print(f"  {path}  (epoch={ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f})")

print()
print("Note: these results are on CLEAN validation data.")
print("Notebook 06 evaluates all models on CORRUPTED data")
print("-- that is where ARAF should show its advantage.")


## 10. Load best checkpoints

We reload the best checkpoint for each model (not necessarily the last epoch)
and verify they load correctly. Notebook 06 will import these directly.


In [ ]:
def load_best_models():
    """Load best checkpoint for each model."""
    best_models = build_models()  # fresh model with same architecture

    for name, model in best_models.items():
        path = best_ckpt_path[name]
        if os.path.exists(path):
            ckpt = torch.load(path, map_location=DEVICE)
            model.load_state_dict(ckpt["state_dict"])
            model.eval()
            print(f"Loaded {name} from epoch {ckpt['epoch']} "
                  f"(val_acc={ckpt['val_acc']:.4f})")
        else:
            print(f"WARNING: No checkpoint found for {name} at {path}")

    return best_models

best_models = load_best_models()
print()
print("All best models loaded and ready for evaluation.")


## 11. Quick sanity check — trained model predictions

Before we move to formal evaluation, let's look at what the trained
models actually predict on a few samples. This is the human sanity check:
do the predictions look reasonable?


In [ ]:
def show_predictions(models_dict, val_dataset, n=4):
    """Show model predictions on a few validation samples."""
    print(f"{'Question':<35} {'Label':>10} {'ImgModel':>10} "
          f"{'TxtModel':>10} {'Naive':>10} {'ARAF':>10}")
    print("-" * 90)

    for i in range(n):
        sample_dict = val_dataset[i]
        batch = {k: v.unsqueeze(0) if isinstance(v, torch.Tensor) else [v]
                 for k, v in sample_dict.items()}
        batch["image"]          = sample_dict["image"].unsqueeze(0)
        batch["text_ids"]       = sample_dict["text_ids"].unsqueeze(0)
        batch["attention_mask"] = sample_dict["attention_mask"].unsqueeze(0)
        batch["label"]          = sample_dict["label"].unsqueeze(0)

        true_ans = idx2answer.get(
            sample_dict["label"].argmax().item(), "?")
        q = sample_dict["raw_text"][:33] + ".." if len(
            sample_dict["raw_text"]) > 35 else sample_dict["raw_text"]

        preds = {}
        for name, model in models_dict.items():
            model.eval()
            with torch.no_grad():
                out = model(batch)
                pred_idx = out["logits"].argmax(dim=1).item()
                preds[name] = idx2answer.get(pred_idx, "?")

        print(f"{q:<35} {true_ans:>10} "
              f"{preds['UnimodalImage']:>10} "
              f"{preds['UnimodalText']:>10} "
              f"{preds['NaiveFusion']:>10} "
              f"{preds['ARAF']:>10}")

show_predictions(best_models, val_dataset, n=6)
print()
print("Note: with 5 epochs on 2000 samples, predictions will often be wrong.")
print("This is expected. With more data and epochs, accuracy improves significantly.")
print("The important comparison is ARAF vs NaiveFusion under corruption (Notebook 06).")


## 12. Save training history for Notebook 06


In [ ]:
# Save history as JSON so Notebook 06 can load it for combined plots
history_serializable = {}
for name, h in history.items():
    history_serializable[name] = {k: [float(v) for v in vals]
                                   for k, vals in h.items()}

with open("training_history.json", "w") as f:
    json.dump(history_serializable, f, indent=2)

print("Saved: training_history.json")
print("Saved: checkpoints/<model>_best.pt  (one per model)")
print()
print("Your project folder now:")
print("  checkpoints/")
print("    UnimodalImage_best.pt")
print("    UnimodalText_best.pt")
print("    NaiveFusion_best.pt")
print("    ARAF_best.pt")
print("  training_history.json")
print("  training_curves.png")


## 13. Summary and what's next

### What we did in this notebook

- Built `VQACorruptionDataset` — applies corruption on-the-fly per epoch
- Trained all four models with identical setup for fair comparison
- Used Adam optimizer with cosine learning rate decay
- Saved best checkpoints based on clean validation accuracy
- Visualized training curves and loss decomposition

### Key things to remember about these results

The clean validation accuracy here is not ARAF's main claim.
ARAF is designed to be robust under corruption, not necessarily
to get the highest clean accuracy (though it should be competitive).

The real evaluation is in Notebook 06, where we:
- Load the trained models from checkpoints
- Run them on corrupted validation data
- Plot accuracy vs corruption severity curves
- Show that ARAF degrades less than NaiveFusion under corruption
- Show reliability scores on trained model (the heatmap from Notebook 04)

That is the figure that goes in your paper.

### Scaling up for paper results

When you're ready to generate final results:
1. Set `MAX_TRAIN_SAMPLES = None` (full dataset)
2. Set `NUM_EPOCHS = 20` (or more)
3. Consider unfreezing the last 2 layers of ResNet and BERT
   (add `frozen_encoders=False` and use a smaller LR of 1e-5 for encoders)
4. Run on GPU if available

This will take several hours but the accuracy numbers will be
publication-quality.
